# Training Research Notebook

**PR 43 · Step 2A.** Verify frozen inputs and replay the historical abstention search. All comparisons use the PR 39 random split. Target-derived features can use later contacts’ labels, and the MLP early-stopping validation shared encoder fitting. Forward-in-time performance requires separate temporal validation.

In [ ]:
# CELL 1: RESEARCH SETUP
import sys
from pathlib import Path

here = Path.cwd().resolve()
bank_root = next(p for p in (here, *here.parents)
                 if (p / "feature_research/training_inputs.py").is_file())
sys.path.insert(0, str(bank_root))

from feature_research.training_side import draw_training_side
from feature_research.training_inputs import frozen_inputs, record_inputs

## Training population

Draw the accepted stratified split using row identifiers. Only training features and labels are selected. The complement's row identifiers contribute to the router split fingerprint. The Stage I/O split fingerprint is attested from PR 39.

In [ ]:
# CELL 2: TRAINING POPULATION
TRAIN = draw_training_side(indices_only=True)

## Frozen OOF inputs and reference configuration

Stages 1 and 3 provide positive-subscription probabilities (sigmoid at Stage 1, fold-gated isotonic at Stage 3). Stage 3 raw scores are retained separately. Each router supplies its original route, confidence, transformed score, and producing fold's base rate. On Pass 1 confidence measures negative precision, on Pass 2 positive precision. GLASS uses the first matching rule's fit-sample precision; RF uses inner-OOF band precision. On deferral the score equals the fold base rate and confidence is zero. Deferral carries no router vote; Meta-XGBoost masks the probability in this state.

PR 39 GLASS voter cuts target 0.70 recall and use hybrid weights (blend 0.5). Meta-XGBoost uses raw meta OOF probabilities and fold-nested class cuts. The replay fixes the accepted configurations and calls the production abstention search: retain confidence ≥ cut, require coverage ≥ 0.50, maximize retained F2 across distinct observed confidences, and choose the smaller cut on ties. The older research helper scans a fixed grid. These historical settings are references for later registration.

In [ ]:
# CELL 3: VERIFY UPSTREAM INPUTS AND HISTORICAL SEARCH
OOF, evidence = frozen_inputs(TRAIN)
for arm, table in OOF.items():
    print(arm, table.shape, table.route.value_counts().to_dict())
for name, reference in evidence["historical_reference"].items():
    print(name, reference["configuration"], reference["replay"])

## Research record

Save verified identities, accepted configurations, and replay results. Per-row tables remain in memory for later research. Step 2B registers policy and Step 2C adds nested Stage 4 validation.

In [ ]:
# CELL 4: WRITE STEP 2A EVIDENCE
record_path = record_inputs(evidence)
print(record_path)

## Step 2B — Register the Stage 4 decision policy

After the setup cell, run Cell 5 directly to register against the accepted Step 2A record. Cells 2–4 replay the historical inputs and are unnecessary for this step. The versioned policy below defines the research costs, coverage obligations, comparisons, budgets, and evidence required for promotion. The receipt ties its digest to the verified frozen-input record.

In [ ]:
# CELL 5: REGISTER THE STAGE 4 POLICY
from pprint import pprint
from feature_research.stage4_policy import load_policy, register

policy = load_policy()
registration_path = register()
print(registration_path)
pprint(policy, sort_dicts=False)

## Step 2C — Nested Stage 4 references on fixed upstream inputs

Run after Cells 1–3; Cells 4–5 need not be repeated. The five outer folds are the accepted shared producer folds. Inside each outer training partition, three stratified inner folds (seed 42) produce the predictions that select every learned Stage 4 quantity: GLASS voter cuts and weights, the Meta-XGBoost proposal (the 12 registered RandomSampler proposals, historical F2-at-0.5 objective, raw space), class thresholds and deferral cuts. The selected procedure is refitted on the outer training partition, locked, and scores the untouched outer rows once. Each procedure runs under the historical rule (retained F2, coverage ≥ 0.50) and the registered whole-system policy. The accepted PR 39 Meta-XGBoost hyperparameters are a descriptive reference only.

Before any search, the run verifies the policy, its receipt and the Step 2A record, and reproduces the accepted PR 39 cuts, weights and meta OOF probabilities from the same inputs. Optimism is outer achieved loss minus inner selected loss, weighted by outer rows. The record lists the digest of every stored score; the scores themselves (`nested_predictions.npz`, compressed) stay local beside it, for reuse in Step 4B.

Upstream OOF inputs stay fixed, and the upstream fits behind a row's inputs can include other outer rows' labels. This is nested Stage 4 evaluation, not nested cascade evaluation (PR 44). Inner-to-outer differences also reflect sampling variation and fitting-set size. The evidence is random-split and training-only; temporal validation is PR 49.

In [ ]:
# CELL 6: NESTED STAGE 4 REFERENCES
from feature_research.stage4_nested import nested_references, record_nested, summarize

NESTED, nested_evidence, nested_local = nested_references(OOF, evidence)
display(summarize(nested_evidence).round(4))
nested_paths = record_nested(nested_evidence, nested_local)
print(*nested_paths, sep="\n")

## Commit 3 — Voter diagnostics and GLASS decision structure

Run the diagnostic command once, set `PR43_VOTER_RUN` to its run ID, and execute Cell 7 after setup. The first tables describe the frozen PR 39 GLASS operating point. Compare the two fold-refitted rules separately. A sole correct voter is counted among active votes. A correct lone dissent requires all three. Exact flag budgets preserve whole ties.

The negative-margin ceiling takes negative LR/EBM probabilities to zero with all votes and positive evidence held fixed. A forced positive remains positive even at that ceiling. Calibration lines use a joint intercept and slope. Each optional route variant requires a qualifying contrast on its own outer training rows. Row-level uncertainty is conditional on the frozen random split.

In [ ]:
# CELL 7: COMMIT 3 VOTER DIAGNOSTICS
import os
import pandas as pd
from feature_research.research_records import read_run_record
from feature_research.stage4_policy import LOGS

report = read_run_record(os.environ['PR43_VOTER_RUN'], 'voter_diagnostics', root=LOGS)
hist = report['historical_glass']
display(pd.DataFrame(hist['contingency']['joint']))
for name in ('pairs', 'unique'):
    display(pd.DataFrame(hist['contingency'][name]).query("label == 'all' and route == 'all'"))
display({k: v['brackets'] for k, v in hist['matched'].items()}, hist['router_tradeoffs'])
display(pd.DataFrame(report['deferred_quality']), pd.DataFrame(report['calibration']))
comp = pd.DataFrame(hist['composition'])
display(comp.groupby(['route','y','retained','correct']).n.sum().unstack(fill_value=0))
display(pd.DataFrame(hist['margin_bound']), pd.DataFrame(report['nested_glass_summary']))
comp.groupby(['route','retained']).n.sum().unstack(fill_value=0).plot.bar(stacked=True, ylabel='Training rows')
display(pd.DataFrame(report['route_coefficients'])[['upstream','outer_fold','status']])
display(report['handoff'])
print(report['rule_evidence'], report['joint_corners'], report['commit5_expansion'], sep='\n')

## Step 4A — GLASS Linear Arbiter

Run the research command once, set `PR43_LINEAR_RUN` to the resulting run ID, then run Cell 8 after setup. The log-odds are a free intercept plus three nonnegative slopes times signed voter evidence. LR/MLP and EBM/XGB evidence is the logit of the positive probability minus the logit of its fold-fitted recall-0.70 cut. Both router transforms recover the routed positive rate using the producing fold's base rate. Subtract the base-rate logit and use exactly zero on deferral. Logit inputs are clipped to [0.000001, 0.999999].

Four ridge penalties apply to mean log-loss and exclude the intercept. The conditional model uses a shared intercept and shrinks route slopes toward its fitted global anchor. Each eligible form selects a penalty and a locked operating cut on inner predictions. A nondecreasing sigmoid of the additive logit competes with identity by inner log-loss. Three subfolds per inner fitting set isolate calibration. The slope measures change per unit of **log-odds evidence**. Its realized contribution also depends on the evidence on that row.

The four primary contrasts use the inner-selected procedure for each upstream. Route versus global is descriptive. Fold coefficients show stability. Conditional coefficient intervals appear only for a provisionally retained specification; they exclude upstream refitting and reselection uncertainty.

In [ ]:
# CELL 8: STEP 4A LINEAR ARBITER
import os
import pandas as pd
from feature_research.research_records import read_run_record
from feature_research.stage4_policy import LOGS

linear = read_run_record(os.environ['PR43_LINEAR_RUN'], 'linear_arbiter', root=LOGS)
for upstream, result in linear['results'].items():
    print(upstream, result['disposition'])
    if 'failure' in result:
        print(result['failure'])
        continue
    chosen = result['procedures']['selected']
    display(pd.DataFrame([{k: v for k, v in quality.items() if k != 'calibration'}
                          for quality in chosen['probability'].values()], index=chosen['probability']))
    display(pd.DataFrame([dict(reference=name, **c['interval'], verdict=c['loss_verdict'],
                               failed=[k for k, ok in c['safeguards'].items() if not ok])
                          for name, c in result['reference_comparisons'].items()]))
    coefficients = []
    for fold in result['selection']:
        for form, cfg in fold['configurations'].items():
            labels = ['intercept'] + ([v for v in ('stage1','stage3','router')] if form == 'global' else
                                     [s+' '+v for s in ('pass1','pass2','uncertain') for v in ('stage1','stage3','router')])
            coefficients.append(dict(fold=fold['fold'], form=form, selected=form == fold['selected'],
                                     penalty=cfg['penalty'], **dict(zip(labels,cfg['beta']))))
    display(pd.DataFrame(coefficients), result['coefficient_intervals'])
    if result['route_minus_global_descriptive']:
        display(result['route_minus_global_descriptive']['interval'], result['route_evidence'])
    risks = {name: values['pooled']['matched_risk_row_weighted'] for name, values in result['procedures'].items()}
    pd.DataFrame(risks).reindex(['0.5','0.7','0.85','1']).plot(marker='o', title=upstream,
        xlabel='Exact matched coverage', ylabel='Selective risk')

## Step 4B — GLASS formula and Meta-XGBoost comparators

Set `PR43_COMPARATOR_RUN` to the accepted comparator run and execute Cell 9 after setup. Compare the normalized and reference margins first. Both use the same fitted weights and cuts. Each Meta objective ranks the same stored proposals. Its two operating spaces share a winner and use independently inner-selected cuts. The accepted parameter reference stays descriptive in raw space.

The curve shows risk at the smallest attainable coverage meeting each obligation. Locked-cut outcomes and exact matched-coverage contrasts are recorded separately. Step 4B intervals are descriptive. The four adjusted primary comparisons and Linear Arbiter dispositions remain in the linked Step 4A record.

In [ ]:
# CELL 9: STEP 4B COMPARATOR RESULTS
import os
import pandas as pd
from feature_research.research_records import read_run_record
from feature_research.stage4_policy import LOGS

comparators = read_run_record(os.environ['PR43_COMPARATOR_RUN'], 'stage4_comparators', root=LOGS)
print(comparators['rule_evidence'])
for upstream, data in comparators['results'].items():
    print(upstream, 'Linear Arbiter:', data['linear_disposition'])
    display(pd.DataFrame([dict(fold=d['fold'], joint=d['formula']['corners'], formula=d['formula']['selected'],
        meta=d['meta']['selected'], f2=d['meta']['f2_winner'], logloss=d['meta']['logloss_winner'])
        for d in data['design']]))
    display(pd.DataFrame({name:arm['pooled']['outer'] for name,arm in data['arms'].items()
                          if 'pooled' in arm}).T[['coverage','loss','effective_recall','positive_retention','negative_retention']])
    display(pd.DataFrame({name:dict(result=c['secondary_result'], **c['interval'])
                          for name,c in data['secondary'].items()}).T)
    pd.DataFrame(data['minimum_coverage_risk'], index=data['coverage_axis'])[
        ['formula_reference','formula_selected','meta_f2_raw','meta_selected','linear']].plot(
        title=upstream, xlabel='Minimum coverage', ylabel='Selective risk')